# Phase 2 Notebook: Ingestion + Canonical Identity

## What was done
- Implemented fail-soft ingestion adapters with schema validation and immutable raw snapshots.
- Added provenance metadata sidecars with refresh timestamps and upstream references.
- Implemented canonical player mapping with candidate restriction (team/position) and confidence scoring.
- Added point-in-time joins for temporal consistency across sources.

## Why it was done
- To prevent source outages from breaking the weekly pipeline.
- To keep identity mapping auditable and reduce false-positive joins.
- To avoid temporal leakage by using point-in-time aligned reference data.

In [1]:
import pandas as pd
from datetime import datetime, timezone

left = pd.DataFrame([
    {'left_id': 'fpl_1', 'player_name': 'Bukayo Saka', 'team': 'ARS', 'position': 'MID'},
    {'left_id': 'fpl_2', 'player_name': 'Erling Haaland', 'team': 'MCI', 'position': 'FWD'},
])
right = pd.DataFrame([
    {'right_id': 'ud_10', 'player_name': 'B. Saka', 'team': 'ARS', 'position': 'MID'},
    {'right_id': 'ud_20', 'player_name': 'E. Haaland', 'team': 'MCI', 'position': 'FWD'},
    {'right_id': 'ud_99', 'player_name': 'Bukayo Saka', 'team': 'MCI', 'position': 'MID'},
])
left, right

(  left_id     player_name team position
 0   fpl_1     Bukayo Saka  ARS      MID
 1   fpl_2  Erling Haaland  MCI      FWD,
   right_id  player_name team position
 0    ud_10      B. Saka  ARS      MID
 1    ud_20   E. Haaland  MCI      FWD
 2    ud_99  Bukayo Saka  MCI      MID)

## Data quality checks
- Required key presence for each source record.
- Source freshness against staleness thresholds.
- Mapping confidence distribution and unmatched rate.
- Team/position consistency checks for matched entities.

In [2]:
required_keys = {'player_name', 'team', 'position'}
left_missing = [idx for idx, row in left.iterrows() if not required_keys.issubset(set(row.index))]
right_missing = [idx for idx, row in right.iterrows() if not required_keys.issubset(set(row.index))]

quality_report = {
    'left_rows': len(left),
    'right_rows': len(right),
    'left_missing_required_rows': len(left_missing),
    'right_missing_required_rows': len(right_missing),
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
}
quality_report

{'left_rows': 2,
 'right_rows': 3,
 'left_missing_required_rows': 0,
 'right_missing_required_rows': 0,
 'generated_at_utc': '2026-07-28T20:25:19.045384+00:00'}

## Findings and anomalies
- Candidate `ud_99` has the same name as Bukayo Saka but a different team, so team-restricted matching prevents a false positive.
- Abbreviated names (for example `B. Saka`) can still match correctly with high confidence when team and position are aligned.

## How anomalies were handled
- Team and/or position restriction is applied before fuzzy scoring.
- Low-confidence candidates are rejected using a configurable score cutoff.
- Stale or broken sources are marked and do not block the rest of ingestion (fail-soft).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

left_eda = left.copy()
right_eda = right.copy()

left_eda["name_length"] = left_eda["player_name"].str.len()
right_eda["name_length"] = right_eda["player_name"].str.len()
left_eda["is_abbrev"] = left_eda["player_name"].str.contains(r"\\.", regex=True).astype(int)
right_eda["is_abbrev"] = right_eda["player_name"].str.contains(r"\\.", regex=True).astype(int)

team_counts = right_eda["team"].value_counts().sort_index()
position_counts = right_eda["position"].value_counts().sort_index()
abbrev_rates = pd.Series(
    {
        "left_source": float(left_eda["is_abbrev"].mean()),
        "right_source": float(right_eda["is_abbrev"].mean()),
    }
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(team_counts.index, team_counts.values, color="#4e79a7")
axes[0].set_title("Right Source: Team Distribution")
axes[0].set_xlabel("Team")
axes[0].set_ylabel("Rows")

axes[1].bar(position_counts.index, position_counts.values, color="#59a14f")
axes[1].set_title("Right Source: Position Distribution")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Rows")

axes[2].bar(abbrev_rates.index, abbrev_rates.values, color="#f28e2b")
axes[2].set_title("Abbreviated Name Rate")
axes[2].set_ylim(0, 1)
axes[2].set_ylabel("Share of Rows")

plt.tight_layout()
plt.show()

eda_summary = {
    "left_avg_name_length": round(float(left_eda["name_length"].mean()), 2),
    "right_avg_name_length": round(float(right_eda["name_length"].mean()), 2),
    "left_abbrev_rate": round(float(abbrev_rates["left_source"]), 3),
    "right_abbrev_rate": round(float(abbrev_rates["right_source"]), 3),
}
eda_summary